In [ ]:
# === SatQuery AI: Pipeline B — Bi-Temporal Change VQA (CDVQA) ===
!pip install -q -U transformers accelerate peft bitsandbytes datasets pillow huggingface_hub



In [ ]:
import os, json, torch, random
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoProcessor,
    LlavaForConditionalGeneration,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from huggingface_hub import HfApi, login

HF_TOKEN = os.environ.get('HF_TOKEN', '')
if HF_TOKEN:
    login(token=HF_TOKEN)

print('CUDA available:', torch.cuda.is_available())



In [ ]:
# === Create Bi-Temporal Paired Dataset ===
os.makedirs('bitemporal_samples', exist_ok=True)
sample_pairs = []

CHANGE_TYPES = [
    ('new_construction', 'New residential and commercial structures have been constructed in the central quadrant.'),
    ('deforestation', 'Significant clearing of dense tree canopy is evident between T1 and T2.'),
    ('vegetation_growth', 'Substantial expansion in agricultural greenness and seasonal crop vigour is observed.')
]

for i in range(150):
    ct, desc = random.choice(CHANGE_TYPES)
    t1_img = Image.new('RGB', (224, 224), (random.randint(40, 70), random.randint(80, 120), random.randint(40, 70)))
    t2_img = Image.new('RGB', (224, 224), (random.randint(80, 130), random.randint(60, 90), random.randint(50, 80)))
    
    combined = Image.new('RGB', (448, 224))
    combined.paste(t1_img, (0, 0))
    combined.paste(t2_img, (224, 0))
    
    path = f'bitemporal_samples/pair_{i}.jpg'
    combined.save(path)
    sample_pairs.append({
        'image_path': path,
        'question': 'What changes have occurred between date 1 (left) and date 2 (right)?',
        'answer': f'Comparative analysis reveals {ct.replace("_", " ")}: {desc}'
    })

print(f'Generated {len(sample_pairs)} bi-temporal change QA pairs.')



In [ ]:
# === Load 4-Bit Base Model ===
BASE_MODEL = 'llava-hf/llava-1.5-7b-hf'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

processor = AutoProcessor.from_pretrained(BASE_MODEL)
model = LlavaForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16
)

model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()



In [ ]:
# === Training Loop ===
class BiTemporalDataset(Dataset):
    def __init__(self, samples, processor):
        self.samples = samples
        self.processor = processor

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        image = Image.open(item['image_path']).convert('RGB')
        prompt = f"USER: <image>\n{item['question']}\nASSISTANT: {item['answer']}"
        inputs = self.processor(text=prompt, images=image, return_tensors='pt', padding='max_length', max_length=128, truncation=True)
        return {k: v.squeeze(0) for k, v in inputs.items()}

train_dataset = BiTemporalDataset(sample_pairs, processor)

train_args = TrainingArguments(
    output_dir='./results_change_vqa',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=10,
    fp16=True,
    save_strategy='no',
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=train_dataset
)

print('Starting Bi-Temporal Change-VQA QLoRA fine-tuning...')
trainer.train()
print('Training completed!')



In [ ]:
# === Save & Push to Hugging Face Hub ===
SAVE_DIR = './satquery_ai_change_vqa'
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

HF_REPO = 'mokshda/satquery-ai-change-vqa'
try:
    api = HfApi()
    api.create_repo(repo_id=HF_REPO, exist_ok=True, private=False)
    api.upload_folder(
        folder_path=SAVE_DIR,
        repo_id=HF_REPO,
        repo_type='model'
    )
    print(f'Successfully uploaded adapter weights to https://huggingface.co/{HF_REPO}')
except Exception as e:
    print('Upload error (check HF_TOKEN):', e)

